In [1]:
import json
import os
import sqlite3

from google import genai
from google.genai import types

DB_PATH = "data.nosync/recipes.db"
# Model -> requests per minute allowed per key. Every key x model pair has its own quota,
# so 2 keys x 2 models at 14 RPM = 56 requests/min. Check the limits in AI Studio and adjust.
MODELS = {
    "gemini-3.5-flash-lite": 15,  # paid tier allows 4K RPM / 4M TPM; at ~7K tokens per request TPM caps us near 550
    "gemini-3.1-flash-lite": 15,  # often overloaded (503) as of 2026-10-28
}
BATCH_SIZE = 200

# The two new columns. Rename here if you called them something else.
# processed: 0 = not yet, 1 = done, -1 = failed validation twice (look at these by hand)
COL_PROCESSED = "ingredients_processed"
COL_CLEAN = "ingredients_clean"  # JSON array of English ingredient names

# One API key per line in gemini_keys.txt (git-ignored, so the keys never land in a commit).
# Quotas are per Google Cloud project, so two keys only help if they come from two different projects.
with open("gemini_keys.txt") as f:
    API_KEYS = [line.strip() for line in f if line.strip() and not line.startswith("#")]
API_KEYS = list(dict.fromkeys(API_KEYS))  # the same key twice would just share one quota
clients = [genai.Client(api_key=key) for key in API_KEYS]

# timeout: wait instead of failing if the crawler is writing at the same moment
con = sqlite3.connect(DB_PATH, timeout=60)
cols = {row[1] for row in con.execute("PRAGMA table_info(recipes)")}
if COL_PROCESSED not in cols:
    con.execute(f"ALTER TABLE recipes ADD COLUMN {COL_PROCESSED} INTEGER NOT NULL DEFAULT 0")
if COL_CLEAN not in cols:
    con.execute(f"ALTER TABLE recipes ADD COLUMN {COL_CLEAN} TEXT")
# Without this index every batch would scan the whole multi-GB table.
con.execute(f"CREATE INDEX IF NOT EXISTS recipes_{COL_PROCESSED} ON recipes({COL_PROCESSED})")
con.commit()

In [2]:
import threading
import time

PROMPT = """You extract ingredients from recipes.

The input is a JSON array of recipes, each {"id": <number>, "ingredients": [<ingredient lines>]}.
The lines can be in any language and contain quantities, units, notes and section headings.

For every recipe, return {"id": <same id>, "ingredients": [<names>]}. Return exactly one entry per input id.

Rules for each name:
- Translate it into English, lowercase, singular ("eggs" -> "egg", "Zwiebeln" -> "onion").
- Only the ingredient itself: no quantities, units, sizes, brands or notes.
- Remove preparation and state words: organic, fresh, frozen, canned, dried, ground, minced, chopped, sliced,
  peeled, grated, shredded, crushed, mashed, beaten, whipped, roasted, toasted, smoked, pickled, fermented,
  marinated, seasoned, spiced, herbed, flavored.
- Keep words that make it a different ingredient: "brown sugar", "egg white", "soy sauce", "coconut milk".
- If a line offers alternatives ("chicken breasts or thighs"), return only the first one.
- Skip anything that is not food: equipment, section headings ("For the sauce:"), serving tips.
- If a line contains several ingredients ("salt and pepper"), return each one separately.

Example input:  [{"id": 0, "ingredients": ["2 cups chopped onions", "Salz und Pfeffer", "Für die Soße:", "1 EL Sojasauce"]}]
Example output: [{"id": 0, "ingredients": ["onion", "salt", "pepper", "soy sauce"]}]

Input:
"""

RESPONSE_SCHEMA = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "id": {"type": "integer"},
            "ingredients": {"type": "array", "items": {"type": "string"}},
        },
        "required": ["id", "ingredients"],
    },
}


def fetch_batch(n=BATCH_SIZE):
    """Up to n unprocessed recipes as [(rowid, [ingredient lines])]."""
    rows = con.execute(
        f"SELECT rowid, ingredients FROM recipes WHERE {COL_PROCESSED} = 0 LIMIT ?", (n,)
    ).fetchall()
    return [(rowid, json.loads(ing)) for rowid, ing in rows]


from google.genai import errors


class Lane:
    """One API key + one model, with its own rate limit."""

    def __init__(self, key_no, client, model, rpm):
        self.name = f"key{key_no}/{model}"
        self.client, self.model = client, model
        self.interval = 1.05 * 60 / rpm  # 5% margin for timing jitter
        self.next_at = 0.0  # earliest start of the next request
        self.backoff = 0.0  # current pause after a 429, doubles while they keep coming


LANES = [Lane(k + 1, c, m, rpm) for k, c in enumerate(clients) for m, rpm in MODELS.items()]
TOTAL_RPM = sum(MODELS.values()) * len(clients)
_lanes_lock = threading.Lock()


def take_lane():
    """Reserve a slot on whichever lane is free soonest, wait for it, return the lane."""
    with _lanes_lock:
        now = time.monotonic()
        lane = min(LANES, key=lambda l: l.next_at)
        start = max(now, lane.next_at)
        lane.next_at = start + lane.interval
    time.sleep(start - now)
    return lane


def call_model(batch):
    """Send a batch; returns {position: [names]} for every entry the model returned."""
    lane = take_lane()
    payload = [{"id": i, "ingredients": lines} for i, (_, lines) in enumerate(batch)]
    try:
        response = lane.client.models.generate_content(
            model=lane.model,
            contents=PROMPT + json.dumps(payload, ensure_ascii=False),
            config=types.GenerateContentConfig(
                temperature=0,
                response_mime_type="application/json",
                response_json_schema=RESPONSE_SCHEMA,
            ),
        )
    except errors.APIError as e:
        if e.code == 429:  # quota used up: rest this lane (1, 2, 4 ... 30 min), the others carry on
            with _lanes_lock:
                lane.backoff = min(max(lane.backoff * 2, 60), 1800)
                lane.next_at = max(lane.next_at, time.monotonic() + lane.backoff)
            print(f"{lane.name}: quota exceeded, pausing {lane.backoff / 60:.0f} min")
        raise
    lane.backoff = 0.0
    return {item["id"]: item["ingredients"] for item in json.loads(response.text)}


def clean_names(names):
    """Strip, lowercase, drop empty or absurdly long entries, remove duplicates (keeps order)."""
    out = []
    for name in names:
        name = " ".join(str(name).split()).lower()
        if name and len(name) <= 60 and name not in out:
            out.append(name)
    return out


def validate(batch, result):
    """Split into (ok, bad): ok = [(rowid, names)], bad = [(rowid, lines)] to retry."""
    ok, bad = [], []
    for i, (rowid, lines) in enumerate(batch):
        names = clean_names(result.get(i, []))
        if i in result and (names or not lines):
            ok.append((rowid, names))
        else:  # id missing from the answer, or every line dropped
            bad.append((rowid, lines))
    return ok, bad


def save(ok, failed=()):
    with con:
        con.executemany(
            f"UPDATE recipes SET {COL_CLEAN} = ?, {COL_PROCESSED} = 1 WHERE rowid = ?",
            [(json.dumps(names, ensure_ascii=False), rowid) for rowid, names in ok],
        )
        con.executemany(
            f"UPDATE recipes SET {COL_PROCESSED} = -1 WHERE rowid = ?",
            [(rowid,) for rowid, _ in failed],
        )


def run_batch(batch):
    """API call + validation with one retry for failed recipes. No DB access, so it can run in a thread."""
    try:
        ok, bad = validate(batch, call_model(batch))
    except (json.JSONDecodeError, KeyError, TypeError) as e:  # malformed answer: retry everything once
        print("bad response, retrying:", e)
        ok, bad = [], batch
    failed = []
    if bad:  # one retry, only for the recipes that failed
        try:
            retry_ok, failed = validate(bad, call_model(bad))
        except (json.JSONDecodeError, KeyError, TypeError):
            retry_ok, failed = [], bad
        ok += retry_ok
    return ok, failed


def process_one_batch():
    batch = fetch_batch()
    if not batch:
        return 0
    ok, failed = run_batch(batch)
    save(ok, failed)
    print(f"saved {len(ok)}, failed {len(failed)}")
    return len(batch)


In [3]:
if False:
    # One batch, then look at what was written next to the original lines.
    process_one_batch()

    for name, lang, raw, clean in con.execute(
        f"SELECT name, language, ingredients, {COL_CLEAN} FROM recipes WHERE {COL_PROCESSED} = 1 LIMIT 5"
    ):
        print(f"\n{name} ({lang})")
        for line in json.loads(raw):
            print("   ", line)
        print("  ->", json.loads(clean))

In [4]:
import math
from concurrent.futures import ThreadPoolExecutor, as_completed

# Enough threads to keep every lane busy at ~20 s per request; the lanes do the rate limiting.
WORKERS = math.ceil(TOTAL_RPM * 20 / 60) + 2
print(f"{len(LANES)} lanes, {TOTAL_RPM} requests/min, {WORKERS} workers")


def process_parallel(workers=WORKERS):
    """One round: fetch workers * BATCH_SIZE rows, run the batches in parallel, save as they finish.

    Only this (main) thread touches SQLite. A batch whose API call raised (rate limit, network)
    is not saved, so its rows stay at 0 and are picked up again next round.
    """
    rows = fetch_batch(workers * BATCH_SIZE)
    batches = [rows[i:i + BATCH_SIZE] for i in range(0, len(rows), BATCH_SIZE)]
    saved = errors = 0
    with ThreadPoolExecutor(workers) as pool:
        futures = [pool.submit(run_batch, b) for b in batches]
        for future in as_completed(futures):
            try:
                ok, failed = future.result()
            except Exception as e:
                errors += 1
                print("API error, batch will be retried next round:", e)
                continue
            save(ok, failed)
            saved += len(ok) + len(failed)
    return saved, errors, len(batches)


# Everything else. Safe to interrupt and re-run: it continues with the rows still at 0.
started = time.time()
total = con.execute(f"SELECT COUNT(*) FROM recipes WHERE {COL_PROCESSED} = 0").fetchone()[0]
done = 0
while True:
    saved, errors, n_batches = process_parallel()
    if n_batches == 0:
        break
    if errors == n_batches:  # whole round failed: probably rate limited, back off
        print("every batch failed, waiting 60 s")
        time.sleep(60)
        continue
    done += saved
    rate = done / (time.time() - started)
    print(f"{done:,}/{total:,} recipes, {rate * 3600:,.0f}/h, ~{(total - done) / rate / 3600:.1f} h left")

print(con.execute(f"SELECT {COL_PROCESSED}, COUNT(*) FROM recipes GROUP BY 1").fetchall())


20 lanes, 300 requests/min, 102 workers
API error, batch will be retried next round: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
API error, batch will be retried next round: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
key3/gemini-3.5-flash-lite: quota exceeded, pausing 1 min
API error, batch will be retried next round: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource has been exhausted (e.g. check quota).', 'status': 'RESOURCE_EXHAUSTED'}}
key2/gemini-3.5-flash-lite: quota exceeded, pausing 1 min
API error, batch will be retried next round: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource has been exhausted (e.g. check quota).', 'status': 'RESOURCE_E

KeyboardInterrupt: 